# Multi-cue ablation

Loads committed reference ablation results under `reference_results/ablation/`
(Full = Multi-cue 0.9676). Optionally re-runs `scripts/run_multimodal_ablation.py` on video.


In [1]:
from pathlib import Path
import subprocess, sys
import matplotlib.pyplot as plt
import pandas as pd
plt.switch_backend('Agg')
ROOT = Path('..').resolve()
REF = Path('reference_results') / 'ablation'
OUT = ROOT / 'outputs' / 'ablation_multicue'
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
VARIANT_ORDER=['cue_motion','cue_face','cue_hands','concat','attention_only','gating_only','full']
print('REF', REF.resolve())


REF /Users/jeremiah/Documents/PHD/phd_thesis_joke-Publication -14-oct-complet/analysis/reference_results/ablation


## 1. Reference ablation table


In [2]:
summary_path = REF / 'ablation_summary.csv'
assert summary_path.exists(), summary_path
summary = pd.read_csv(summary_path)
if 'variant' in summary.columns:
    present=[v for v in VARIANT_ORDER if v in set(summary['variant'])]
    if present:
        summary=summary.set_index('variant').loc[present].reset_index()
summary


,variant,Variant,Cues used,Fusion mechanism,Accuracy (%) mean±SD,mean_accuracy,sd_accuracy
0,cue_motion,Motion-only (M),M,—,73.24 ± 3.91,0.732380,0.039086
1,cue_face,Facial-only (F),F,—,86.43 ± 3.44,0.864306,0.034436
2,cue_hands,Gesture-only (G),G,—,96.02 ± 1.45,0.960169,0.014479
3,concat,Naïve fusion (Concat),M+F+G,Concat + MLP,96.20 ± 1.32,0.962000,0.013171
4,attention_only,Cross-modal attention only,M+F+G,Attention (no gating),96.40 ± 1.10,0.964000,0.011015
5,gating_only,Gating only,M+F+G,Scalar gates (no attention),96.55 ± 1.10,0.965500,0.011002
6,full,Full multi-cue (Attention + Gating),M+F+G,Attention + gating (final),96.76 ± 1.11,0.967582,0.011127


In [3]:
acc_col='mean_accuracy'
label_col='variant' if 'variant' in summary.columns else 'Variant'
fig,ax=plt.subplots(figsize=(8,4))
x=range(len(summary))
y=summary[acc_col].to_numpy()
sd=summary['sd_accuracy'].fillna(0).to_numpy() if 'sd_accuracy' in summary.columns else None
colors=['#0072B2' if str(v)=='full' else '#56B4E9' for v in summary[label_col]]
ax.bar(list(x), y, yerr=sd, color=colors, alpha=0.9, capsize=4, width=0.7)
ax.set_xticks(list(x)); ax.set_xticklabels(summary[label_col], rotation=25, ha='right')
ax.set_ylabel('Accuracy'); ax.set_title('Multi-cue ablation')
fig.tight_layout(); fig.savefig(FIG/'multicue_ablation.png', dpi=200); plt.close(fig)
print('Wrote', FIG/'multicue_ablation.png')


Wrote figures/multicue_ablation.png


## 2. Optional video re-run

Set `RUN_VIDEO_ABLATION=True` to overwrite `outputs/ablation_multicue/` from real videos.


In [4]:
RUN_VIDEO_ABLATION=False
SMOKE_MAX_VIDEOS=0
if RUN_VIDEO_ABLATION:
    cmd=[sys.executable,str(ROOT/'scripts'/'run_multimodal_ablation.py'),'--results-dir',str(OUT),'--feature-cache',str(ROOT/'data'/'cache'/'multimodal_features.npz')]
    if SMOKE_MAX_VIDEOS: cmd += ['--max-videos', str(SMOKE_MAX_VIDEOS)]
    print('Running', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)
else:
    print('Using committed reference ablation under', REF)


Using committed reference ablation under reference_results/ablation
